In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.impute import SimpleImputer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, hamming_loss
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import LinearSVC

from sklearn.calibration import CalibratedClassifierCV

import joblib

In [2]:
df = pd.read_csv("cleaned_sleep_apnea_dataset.csv")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (158, 32)

Columns:
['BMI', 'AHI', 'sAHI', 'mean_o2 [%]', 'O2_sats_below_90 [%]', 'lowest_O2(SS) [%]', 'apnoea', 'hypopnoea', 'RSI', 'NOSE', 'Snoring_mouth_closed', 'Snoring_chin_lift', 'AirwaySize_mouth_closed', 'AirwaySize_chin_lift', 'tonsils', 'uvula', 'rhinitis', 'DNS', 'velopharyngeal_collapse [%]', 'oropharyngeal_collapse [%]', 'base_of_tongue_collapse [%]', 'collar_size [cm]', 'STOP_BANG', 'ESS', 'snore_time', 'Flow', 'P', 'T', 'L', 'Tb', 'E', 'treatment_plan_with_stages']


,BMI,AHI,sAHI,mean_o2 [%],O2_sats_below_90 [%],lowest_O2(SS) [%],apnoea,hypopnoea,RSI,NOSE,...,STOP_BANG,ESS,snore_time,Flow,P,T,L,Tb,E,treatment_plan_with_stages
0,25.0,25.8,51.0,91.4,17.5,72.0,14.448044,9.640803248755784,13.0,1.0,...,4.0,10.0,39,0.380035,1,0,0,0,0,1.[continue Rhinitis Rx][Sleep Position Modifi...
1,28.6,36.0,87.0,95.0,10.0,70.0,18.242617,4,5.0,3.0,...,5.0,10.0,0.6875748807704847,0.312668,1,0,1,1,0,1.[Weight Loss]\n2.[Procut Palatoplasty (anter...
2,24.0,29.0,29.0,94.0,2.0,83.0,16.484897,3,6.0,13.0,...,3.0,10.0,0.624196652784825,0.375290,0,0,0,1,0,1.Augmentin for pus L OMC [? weight loss][Rhin...
3,28.5,21.6,40.2,92.8,8.3,78.0,18.328088,12.17310496015483,7.0,8.0,...,2.0,3.0,62,0.301983,1,0,0,2,0,1.[Rhinitis Rx][Sleep Position Modifier][Aller...
4,30.0,24.0,28.0,93.0,10.0,81.0,16.302745,22,6.0,0.0,...,5.0,10.0,0.6850099964262131,0.315372,2,0,0,2,0,1.[Weight Loss] – reduce work on neck at GYM –...


In [3]:
def parse_staged_treatments(text):
    if pd.isna(text):
        return {}

    text = str(text).replace("\r", "\n")
    stage_matches = list(re.finditer(r'(?m)(\d+)\.', text))
    if not stage_matches:
        return {}
    parsed = {}

    for i, match in enumerate(stage_matches):
        stage_num = int(match.group(1))
        start = match.end()
        end = stage_matches[i + 1].start() if i + 1 < len(stage_matches) else len(text)
        block = text[start:end]

        labels = re.findall(r'\[([^\]]+)\]', block)
        labels = [lbl.strip() for lbl in labels if lbl.strip()]

        parsed[stage_num] = labels

    return parsed


df["parsed_plan_raw"] = df["treatment_plan_with_stages"].apply(parse_staged_treatments)

for i in range(3):
    print(f"Row {i}:")
    print(df.loc[i, "treatment_plan_with_stages"])
    print(df.loc[i, "parsed_plan_raw"])
    print("-" * 80)

Row 0:
1.[continue Rhinitis Rx][Sleep Position Modifier][Allergy Testing] +/- MAD trial
2.[RF Palate][Procut Palatoplasty] [RF turbinates + outfracture] [+rpt sleep study]
3.[Barbed Suture] +/- [Septoplasty] after repeat DISE
{1: ['continue Rhinitis Rx', 'Sleep Position Modifier', 'Allergy Testing'], 2: ['RF Palate', 'Procut Palatoplasty', 'RF turbinates + outfracture', '+rpt sleep study'], 3: ['Barbed Suture', 'Septoplasty']}
--------------------------------------------------------------------------------
Row 1:
1.[Weight Loss]
2.[Procut Palatoplasty (anterior / inferior)][Barbed Suture]
3.repeat DISE before any further intervention – look at patients previous DISE
{1: ['Weight Loss'], 2: ['Procut Palatoplasty (anterior / inferior)', 'Barbed Suture'], 3: []}
--------------------------------------------------------------------------------
Row 2:
1.Augmentin for pus L OMC [? weight loss][Rhinitis Rx][Myofunct. Therapy][airgym app]
2.[Chinstrap][Mandibular Advancement Device – if tolerat

In [4]:
# Map similar labels to a single standard name
NORMALIZE_LABELS = {
    "MAD": "Mandibular Advancement Device",
    "Mandibular advancement device": "Mandibular Advancement Device",
    "Mandibular Advancement Device – if tolerated": "Mandibular Advancement Device",
    "Mandibular Advancement Device for snoring": "Mandibular Advancement Device",
    "Mandibular Advancement Device Trial": "Mandibular Advancement Device",

    "continue Rhinitis Rx": "Rhinitis Rx",
    "continue rhinitis Rx": "Rhinitis Rx",
    "continue Rhinitis Rx ": "Rhinitis Rx",
    "Rhinitis therapy": "Rhinitis Rx",
    "Rhinitis therapy ": "Rhinitis Rx",

    "Restart / continue CPAP": "CPAP",
    "continue CPAP": "CPAP",
    "Start CPAP": "CPAP",

    "Procut Palatoplasty (anterior / inferior)": "Procut Palatoplasty",
    "Procut  ant Palatoplasty": "Procut Palatoplasty",
    " anterior Procut Palatoplasty": "Procut Palatoplasty",
    "Palatoplasty (anterior / inferior)": "Procut Palatoplasty",

    "Lingual Tonsil Reduction - coblation": "Lingual Tonsil Reduction",
    "Lingual Tonsil coblation": "Lingual Tonsil Reduction",

    "Hypoglossal N. Stimulator": "Hypoglossal N. Stim",
    "Hypoglossal N. Stim ": "Hypoglossal N. Stim",

    "RF turbinates + outfracture": "RF turbinates + outfracture",
    "RF turbs + out#": "RF turbinates + outfracture",

    "Myofunctional Therapy for snoring": "Myofunct. Therapy",
}


def normalize_label(label):
    label = str(label).strip()
    label = re.sub(r'\s+', ' ', label)  # collapse repeated spaces
    return NORMALIZE_LABELS.get(label, label)


def normalize_plan(plan_dict):
    normalized = {}
    for stage, labels in plan_dict.items():
        clean_labels = [normalize_label(lbl) for lbl in labels]
        # remove duplicates while preserving order
        seen = set()
        deduped = []
        for lbl in clean_labels:
            if lbl not in seen:
                deduped.append(lbl)
                seen.add(lbl)
        normalized[stage] = deduped
    return normalized


df["parsed_plan"] = df["parsed_plan_raw"].apply(normalize_plan)

# Preview
df[["treatment_plan_with_stages", "parsed_plan"]].head()

,treatment_plan_with_stages,parsed_plan
0,1.[continue Rhinitis Rx][Sleep Position Modifi...,"{1: ['Rhinitis Rx', 'Sleep Position Modifier',..."
1,1.[Weight Loss]\n2.[Procut Palatoplasty (anter...,"{1: ['Weight Loss'], 2: ['Procut Palatoplasty'..."
2,1.Augmentin for pus L OMC [? weight loss][Rhin...,"{1: ['? weight loss', 'Rhinitis Rx', 'Myofunct..."
3,1.[Rhinitis Rx][Sleep Position Modifier][Aller...,"{1: ['Rhinitis Rx', 'Sleep Position Modifier',..."
4,1.[Weight Loss] – reduce work on neck at GYM –...,"{1: ['Weight Loss'], 2: ['Mandibular Advanceme..."


In [5]:
stage_counts = defaultdict(Counter)

for plan in df["parsed_plan"]:
    for stage, labels in plan.items():
        stage_counts[stage].update(labels)

for stage in sorted(stage_counts.keys()):
    print(f"\nSTAGE {stage}")
    for label, count in stage_counts[stage].most_common(20):
        print(f"{label:35s} {count}")


STAGE 1
Rhinitis Rx                         92
Weight Loss                         75
Mandibular Advancement Device       45
Sleep Position Modifier             35
Chinstrap                           31
PPI Trial                           20
Allergy Testing                     15
CPAP                                14
CBTi                                13
Myofunct. Therapy                   12
Sleep Hygiene                       10
+rpt sleep study                    5
Tonsillectomy                       4
Weight loss                         3
PPI                                 2
Procut Palatoplasty                 2
NIPF in OPD                         2
allergy testing                     2
Barbed Suture                       2
VitD                                2

STAGE 2
Tonsillectomy                       44
Procut Palatoplasty                 39
Mandibular Advancement Device       34
Barbed Suture                       32
Sleep Position Modifier             28
Chinstrap       

In [6]:
MIN_LABEL_FREQ = 5

common_labels_by_stage = {
    stage: sorted([label for label, count in counts.items() if count >= MIN_LABEL_FREQ])
    for stage, counts in stage_counts.items()
}

print("Common labels by stage:\n")
for stage, labels in common_labels_by_stage.items():
    print(f"Stage {stage}: {labels}")

Common labels by stage:

Stage 1: ['+rpt sleep study', 'Allergy Testing', 'CBTi', 'CPAP', 'Chinstrap', 'Mandibular Advancement Device', 'Myofunct. Therapy', 'PPI Trial', 'Rhinitis Rx', 'Sleep Hygiene', 'Sleep Position Modifier', 'Weight Loss']
Stage 2: ['+rpt sleep study', 'Allergy Testing', 'Barbed Suture', 'Chinstrap', 'Lingual Tonsil Reduction', 'Mandibular Advancement Device', 'Procut Palatoplasty', 'RF BOT', 'RF Palate', 'RF turbinates', 'RF turbinates + outfracture', 'Rhinitis Rx', 'Septoplasty', 'Sleep Position Modifier', 'Tonsillectomy', 'Weight Loss']
Stage 3: ['Barbed Suture', 'Hypoglossal N. Stim', 'Lingual Tonsil Reduction', 'MDT', 'Procut Palatoplasty', 'RF BOT', 'RF turbinates', 'RF turbinates + outfracture', 'Septoplasty', 'Tonsillectomy']
Stage 4: ['Barbed Suture', 'Hypoglossal N. Stim', 'Lingual Tonsil Reduction', 'MDT', 'Procut Palatoplasty', 'RF BOT', 'Tonsillectomy']
Stage 5: []
Stage 6: []


In [7]:
TARGET_COL = "treatment_plan_with_stages"

X = df.drop(columns=["treatment_plan_with_stages", "parsed_plan_raw", "parsed_plan"]).copy()

# Try converting each column to numeric when possible
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="ignore")

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numeric columns:
['BMI', 'AHI', 'sAHI', 'mean_o2 [%]', 'O2_sats_below_90 [%]', 'lowest_O2(SS) [%]', 'apnoea', 'RSI', 'NOSE', 'Snoring_mouth_closed', 'Snoring_chin_lift', 'AirwaySize_mouth_closed', 'AirwaySize_chin_lift', 'velopharyngeal_collapse [%]', 'oropharyngeal_collapse [%]', 'base_of_tongue_collapse [%]', 'collar_size [cm]', 'STOP_BANG', 'ESS', 'Flow']

Categorical columns:
['hypopnoea', 'tonsils', 'uvula', 'rhinitis', 'DNS', 'snore_time', 'P', 'T', 'L', 'Tb', 'E']


C:\Users\Askam Datha Prasad\AppData\Local\Temp\ipykernel_9584\185479571.py:7: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[col] = pd.to_numeric(X[col], errors="ignore")


In [8]:
X_train, X_test, plans_train, plans_test = train_test_split(
    X,
    df["parsed_plan"],
    test_size=0.2,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (126, 31)
Test size: (32, 31)


In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [10]:
stage_models = {}
stage_binarizers = {}
stage_results = []

for stage in sorted(common_labels_by_stage.keys()):
    labels_for_stage = common_labels_by_stage[stage]

    if len(labels_for_stage) == 0:
        continue

    # Builds target labels for each stage
    y_train_lists = plans_train.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])
    y_test_lists = plans_test.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])

    mlb = MultiLabelBinarizer(classes=labels_for_stage)
    Y_train = mlb.fit_transform(y_train_lists)
    Y_test = mlb.transform(y_test_lists)

    # Skip if stage has no usable labels
    if Y_train.shape[1] == 0:
        continue

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", OneVsRestClassifier(
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced"
            )
        ))
    ])

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    micro_f1 = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    hamm = hamming_loss(Y_test, Y_pred)

    stage_models[stage] = model
    stage_binarizers[stage] = mlb

    stage_results.append({
        "stage": stage,
        "n_labels": Y_train.shape[1],
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamm
    })

results_df = pd.DataFrame(stage_results)
results_df

,stage,n_labels,micro_f1,macro_f1,hamming_loss
0,1,12,0.423077,0.299361,0.234375
1,2,16,0.290909,0.153267,0.152344
2,3,10,0.137931,0.121753,0.156250
3,4,7,0.160000,0.112245,0.093750


In [11]:
for _, row in results_df.iterrows():
    print(f"Stage {int(row['stage'])}")
    print(f"  Number of labels : {int(row['n_labels'])}")
    print(f"  Micro F1         : {row['micro_f1']:.3f}")
    print(f"  Macro F1         : {row['macro_f1']:.3f}")
    print(f"  Hamming Loss     : {row['hamming_loss']:.3f}")
    print("-" * 50)

Stage 1
  Number of labels : 12
  Micro F1         : 0.423
  Macro F1         : 0.299
  Hamming Loss     : 0.234
--------------------------------------------------
Stage 2
  Number of labels : 16
  Micro F1         : 0.291
  Macro F1         : 0.153
  Hamming Loss     : 0.152
--------------------------------------------------
Stage 3
  Number of labels : 10
  Micro F1         : 0.138
  Macro F1         : 0.122
  Hamming Loss     : 0.156
--------------------------------------------------
Stage 4
  Number of labels : 7
  Micro F1         : 0.160
  Macro F1         : 0.112
  Hamming Loss     : 0.094
--------------------------------------------------


In [12]:
def predict_treatment_plan(new_data, stage_models, stage_binarizers):
    """
    new_data: DataFrame with same feature columns as X
    returns: dict of stage -> predicted labels
    """
    predictions = {}

    for stage in sorted(stage_models.keys()):
        model = stage_models[stage]
        mlb = stage_binarizers[stage]

        pred_binary = model.predict(new_data)[0]
        pred_labels = [label for label, flag in zip(mlb.classes_, pred_binary) if flag == 1]

        predictions[stage] = pred_labels

    return predictions

In [13]:
# Example: predict for the first row in the test set
example_patient = X_test.iloc[[0]].copy()

predicted_plan = predict_treatment_plan(example_patient, stage_models, stage_binarizers)

print("Predicted plan:")
for stage, labels in predicted_plan.items():
    print(f"Stage {stage}: {labels}")

print("\nActual plan:")
print(plans_test.iloc[0])

Predicted plan:
Stage 1: ['Myofunct. Therapy', 'Rhinitis Rx', 'Weight Loss']
Stage 2: ['Chinstrap', 'RF Palate', 'RF turbinates']
Stage 3: []
Stage 4: []

Actual plan:
{1: ['Rhinitis Rx', 'Sleep Hygiene', 'Allergy Testing']}


In [14]:
def format_predicted_plan(pred_dict):
    lines = []
    for stage in sorted(pred_dict.keys()):
        labels = pred_dict[stage]
        if labels:
            line = f"{stage}. " + " ".join([f"[{lbl}]" for lbl in labels])
            lines.append(line)
    return "\n".join(lines)

print(format_predicted_plan(predicted_plan))

1. [Myofunct. Therapy] [Rhinitis Rx] [Weight Loss]
2. [Chinstrap] [RF Palate] [RF turbinates]


In [15]:
all_predictions = []

for i in range(len(X)):
    row_df = X.iloc[[i]].copy()
    pred = predict_treatment_plan(row_df, stage_models, stage_binarizers)
    pred_text = format_predicted_plan(pred)
    all_predictions.append(pred_text)

df["predicted_treatment_plan"] = all_predictions

df[["treatment_plan_with_stages", "predicted_treatment_plan"]].head(10)

,treatment_plan_with_stages,predicted_treatment_plan
0,1.[continue Rhinitis Rx][Sleep Position Modifi...,1. [Allergy Testing] [Rhinitis Rx] [Sleep Posi...
1,1.[Weight Loss]\n2.[Procut Palatoplasty (anter...,1. [Weight Loss]\n2. [Barbed Suture] [Chinstra...
2,1.Augmentin for pus L OMC [? weight loss][Rhin...,1. [Myofunct. Therapy] [Rhinitis Rx]\n2. [Chin...
3,1.[Rhinitis Rx][Sleep Position Modifier][Aller...,1. [Allergy Testing] [Rhinitis Rx] [Sleep Posi...
4,1.[Weight Loss] – reduce work on neck at GYM –...,1. [Weight Loss]\n2. [Chinstrap] [Mandibular A...
5,1.[Weight Loss][Rhinitis Rx][Sleep Q]\n2.[Chin...,1. [Rhinitis Rx] [Weight Loss]\n2. [Chinstrap]...
6,1.[Weight Loss][Rhinitis Rx]\n2.[RF turbinates...,1. [Rhinitis Rx] [Weight Loss]\n2. [Barbed Sut...
7,1.[Rhinitis Rx]\n2.[MAD],1. [Rhinitis Rx] [Sleep Position Modifier]\n2....
8,1.[Weight Loss][Rhinitis Rx]\n2.[Sleep Positio...,1. [Rhinitis Rx] [Weight Loss]\n2. [Chinstrap]...
9,1.[Weight Loss][Mandibular advancement device]...,1. [Rhinitis Rx] [Weight Loss]\n2. [Tonsillect...


In [16]:
joblib.dump(stage_models, "stage_models.pkl")
joblib.dump(stage_binarizers, "stage_binarizers.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

df.to_csv("sleep_apnea_with_predictions.csv", index=False)

print("Saved:")
print("- stage_models.pkl")
print("- stage_binarizers.pkl")
print("- preprocessor.pkl")
print("- sleep_apnea_with_predictions.csv")

Saved:
- stage_models.pkl
- stage_binarizers.pkl
- preprocessor.pkl
- sleep_apnea_with_predictions.csv


In [17]:
rf_stage_models = {}
rf_stage_binarizers = {}
rf_stage_results = []

for stage in sorted(common_labels_by_stage.keys()):
    labels_for_stage = common_labels_by_stage[stage]

    if len(labels_for_stage) == 0:
        continue

    # Build target labels for each stage
    y_train_lists = plans_train.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])
    y_test_lists = plans_test.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])

    mlb = MultiLabelBinarizer(classes=labels_for_stage)
    Y_train = mlb.fit_transform(y_train_lists)
    Y_test = mlb.transform(y_test_lists)

    if Y_train.shape[1] == 0:
        continue

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", OneVsRestClassifier(
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced_subsample"
            )
        ))
    ])

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    micro_f1 = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    hamm = hamming_loss(Y_test, Y_pred)

    rf_stage_models[stage] = model
    rf_stage_binarizers[stage] = mlb

    rf_stage_results.append({
        "stage": stage,
        "n_labels": Y_train.shape[1],
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamm
    })

rf_results_df = pd.DataFrame(rf_stage_results)
rf_results_df

,stage,n_labels,micro_f1,macro_f1,hamming_loss
0,1,12,0.460317,0.169639,0.177083
1,2,16,0.203390,0.089394,0.091797
2,3,10,0.000000,0.000000,0.075000
3,4,7,0.000000,0.000000,0.044643


In [18]:
svm_stage_models = {}
svm_stage_binarizers = {}
svm_stage_results = []

for stage in sorted(common_labels_by_stage.keys()):
    labels_for_stage = common_labels_by_stage[stage]

    if len(labels_for_stage) == 0:
        continue

    # Build target labels for each stage
    y_train_lists = plans_train.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])
    y_test_lists = plans_test.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])

    mlb = MultiLabelBinarizer(classes=labels_for_stage)
    Y_train = mlb.fit_transform(y_train_lists)
    Y_test = mlb.transform(y_test_lists)

    if Y_train.shape[1] == 0:
        continue

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", OneVsRestClassifier(
            LinearSVC(
                C=1.0,
                class_weight="balanced",
                random_state=42
            )
        ))
    ])

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    micro_f1 = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    hamm = hamming_loss(Y_test, Y_pred)

    svm_stage_models[stage] = model
    svm_stage_binarizers[stage] = mlb

    svm_stage_results.append({
        "stage": stage,
        "n_labels": Y_train.shape[1],
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamm
    })

svm_results_df = pd.DataFrame(svm_stage_results)
svm_results_df

C:\Users\Askam Datha Prasad\anaconda\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Askam Datha Prasad\anaconda\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Askam Datha Prasad\anaconda\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Askam Datha Prasad\anaconda\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Askam Datha Prasad\anaconda\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Askam Datha Prasad\anaconda\Lib\site-packages\sklearn\svm\_base.py:1249

,stage,n_labels,micro_f1,macro_f1,hamming_loss
0,1,12,0.292308,0.123765,0.239583
1,2,16,0.187500,0.044192,0.152344
2,3,10,0.101695,0.017647,0.165625
3,4,7,0.062500,0.011905,0.133929


In [19]:
voting_stage_models = {}
voting_stage_binarizers = {}
voting_stage_results = []

for stage in sorted(common_labels_by_stage.keys()):
    labels_for_stage = common_labels_by_stage[stage]

    if len(labels_for_stage) == 0:
        continue

    # Build target labels for each stage
    y_train_lists = plans_train.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])
    y_test_lists = plans_test.apply(lambda d: [lbl for lbl in d.get(stage, []) if lbl in labels_for_stage])

    mlb = MultiLabelBinarizer(classes=labels_for_stage)
    Y_train = mlb.fit_transform(y_train_lists)
    Y_test = mlb.transform(y_test_lists)

    if Y_train.shape[1] == 0:
        continue

    base_voting_model = VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(
                max_iter=3000,
                class_weight="balanced"
            )),
            ("rf", RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced_subsample"
            )),
            ("svm", CalibratedClassifierCV(
                estimator=LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    random_state=42
                ),
                cv=3
            ))
        ],
        voting="soft",
        n_jobs=-1
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", OneVsRestClassifier(base_voting_model))
    ])

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    micro_f1 = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    hamm = hamming_loss(Y_test, Y_pred)

    voting_stage_models[stage] = model
    voting_stage_binarizers[stage] = mlb

    voting_stage_results.append({
        "stage": stage,
        "n_labels": Y_train.shape[1],
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamm
    })

voting_results_df = pd.DataFrame(voting_stage_results)
voting_results_df

,stage,n_labels,micro_f1,macro_f1,hamming_loss
0,1,12,0.437500,0.169689,0.187500
1,2,16,0.253968,0.102855,0.091797
2,3,10,0.000000,0.000000,0.078125
3,4,7,0.000000,0.000000,0.044643


In [20]:
logistic_results_df = results_df.copy()
logistic_results_df["model"] = "Logistic Regression"
rf_results_df["model"] = "Random Forest"
svm_results_df["model"] = "Linear SVM"
voting_results_df["model"] = "Voting Ensemble"

all_results_df = pd.concat([
    logistic_results_df,
    rf_results_df,
    svm_results_df,
    voting_results_df,
], ignore_index=True)

all_results_df = all_results_df[[
    "model", "stage", "n_labels", "micro_f1", "macro_f1", "hamming_loss"
]]

all_results_df.sort_values(["stage", "micro_f1"], ascending=[True, False])

,model,stage,n_labels,micro_f1,macro_f1,hamming_loss
4,Random Forest,1,12,0.460317,0.169639,0.177083
12,Voting Ensemble,1,12,0.437500,0.169689,0.187500
0,Logistic Regression,1,12,0.423077,0.299361,0.234375
8,Linear SVM,1,12,0.292308,0.123765,0.239583
1,Logistic Regression,2,16,0.290909,0.153267,0.152344
13,Voting Ensemble,2,16,0.253968,0.102855,0.091797
5,Random Forest,2,16,0.203390,0.089394,0.091797
9,Linear SVM,2,16,0.187500,0.044192,0.152344
2,Logistic Regression,3,10,0.137931,0.121753,0.156250
10,Linear SVM,3,10,0.101695,0.017647,0.165625


In [21]:
best_per_stage = all_results_df.sort_values(
    ["stage", "micro_f1"], ascending=[True, False]
).groupby("stage").first().reset_index()

best_per_stage

,stage,model,n_labels,micro_f1,macro_f1,hamming_loss
0,1,Random Forest,12,0.460317,0.169639,0.177083
1,2,Logistic Regression,16,0.290909,0.153267,0.152344
2,3,Logistic Regression,10,0.137931,0.121753,0.156250
3,4,Logistic Regression,7,0.160000,0.112245,0.093750
